# 📚 Shared memory

*Section 14.2 of the notes.*

In notebook `02-processes` we saw the price of processes 📦: every array a
worker needs is converted to bytes, copied into the worker, and rebuilt there.

Think of a class of students who all need to read the same long document.
You could photocopy it for every student 🖨️, or pin one copy on the
whiteboard where everyone can read it. **Shared memory** is the whiteboard: a
block of memory that every process can read directly, so the array is never
copied to the workers.

This notebook shows how to use it, when it pays off, and one more trap of
running many processes: too many threads.

## ⚙️ Setup

The functions run by the workers are again in `tasks.py`, timings use
`%timeit -o -q -r 3 -n 1` with `.best`, and every pool starts its workers with
*spawn*, as in notebook `02-processes`.

As in notebook `02-processes`, the timings use a warm pool,
`tasks.warm_pool(N)`, so that they measure the work and the data transfer, not
the start-up of the workers.

In [ ]:
import multiprocessing
import os
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import shared_memory

import numpy as np

import tasks

N = os.cpu_count()
CTX = multiprocessing.get_context("spawn")  # start workers as fresh Pythons
print("workers:", N)

## 1. 🧱 Putting an array in shared memory

Three steps put an array on the whiteboard:

1. `shared_memory.SharedMemory(create=True, size=...)` asks the operating
   system for a block of memory of the given size in bytes. The block gets a
   **name**, such as `psm_4a5843b9`, which other processes use to find it.
2. `np.ndarray(shape, dtype=..., buffer=shm.buf)` creates a NumPy array that
   *uses the block as its memory*. Nothing is allocated: the array is a view
   of the block, like the views of Chapter 13.
3. `shared[:] = data` copies the numbers into the block, **once**.

In [ ]:
rng = np.random.default_rng(0)
data = rng.standard_normal(20_000_000)  # 160 MB in this process

shm = shared_memory.SharedMemory(create=True, size=data.nbytes)
shared = np.ndarray(data.shape, dtype=data.dtype, buffer=shm.buf)
shared[:] = data  # copied once, into the block

print("name  ", shm.name)
print("size  ", shared.size)
print("shape ", shared.shape)

## 2. ✂️ One piece per worker

We split the array into `N` pieces, one per worker, and compute
`tasks.harmonics` on each piece, the expensive job of the previous notebook.

With shared memory a job does **not** contain its piece of the array. It
contains a small tuple, with everything needed to find the piece on the
whiteboard:

`(name of the block, shape, data type, start, stop)`

Inside the worker, `tasks.harmonics_shared` opens the block by its name, wraps it in
a NumPy array exactly as in Section 1, takes the slice `[start:stop]`, which
is a view and copies nothing, and computes on it. Open `tasks.py` to see the
few lines that do this.

The next cell prepares the jobs in three steps:

1. **where the pieces start and stop**: `np.linspace(0, size, N + 1)` gives
   `N + 1` equally spaced positions from the first entry to the end of the
   array, and piece `i` runs from position `i` to position `i + 1`;
2. **the jobs for shared memory**: one small tuple per piece;
3. **the jobs for sending**: the pieces themselves, which are copied to the
   workers.

It then prints one job of each kind, so that you can see what a worker
receives.

In [ ]:
# Step 1: where the pieces start and stop
edges = np.linspace(0, shared.size, N + 1, dtype=int)
starts = edges[:-1]  # the first entry of every piece
stops = edges[1:]  # one past the last entry of every piece

# Step 2: the jobs for shared memory, a small tuple per piece
jobs_shared = []
for start, stop in zip(starts, stops):
    job = (shm.name, shared.shape, str(shared.dtype), int(start), int(stop))
    jobs_shared.append(job)

# Step 3: the jobs for sending, the pieces themselves
jobs_sent = []
for start, stop in zip(starts, stops):
    jobs_sent.append(data[start:stop])

print("piece 0 runs from", starts[0], "to", stops[0])
print("job 0 for shared memory:", jobs_shared[0])
print(f"job 0 for sending: an array of {jobs_sent[0].nbytes / 1e6:.1f} MB")

The job for shared memory is a few bytes; the job for sending carries
megabytes of numbers. Now we time the three versions:

* 📚 **shared memory**: the workers read their piece from the block;
* 📦 **sending the pieces**: each piece is copied to its worker, as in the
  previous notebook;
* 🐢 **one process**: all the work here, without a pool.

🤔 *Guess: how much faster is shared memory than sending the pieces?*

In [ ]:
with tasks.warm_pool(N) as pool:
    t_shared = %timeit -o -q -r 3 -n 1 list(pool.map(tasks.harmonics_shared, jobs_shared))
    t_sent = %timeit -o -q -r 3 -n 1 list(pool.map(tasks.harmonics_sent, jobs_sent))
t_here = %timeit -o -q -r 3 -n 1 tasks.harmonics(data)

print(f"shared memory       {t_shared.best:5.2f} s")
print(f"sending the pieces  {t_sent.best:5.2f} s")
print(f"one process         {t_here.best:5.2f} s")

💡 **What happened?** Both parallel versions are much faster than one
process 🏎️, and shared memory is faster still. When the pieces are sent, the
whole array is copied once in total (160 MB), each piece into one worker.
That is not much, but the work is short too: the 0.5 s of work is shared by
`N` workers, so each worker computes for only a few hundredths of a second,
about as long as its copy takes.

👉 What matters is **how much is copied in total, compared with the work**.
The next section copies much more.

## 3. 📖 Every worker reads the whole array

Shared memory pays off most when **many tasks read the same large array**.
This happens, for example, in a parameter sweep: every task runs the same
computation on the same data with a different parameter.

Here each of the `N` tasks computes `tasks.harmonics` on the **whole** array.
To keep the memory of all the workers in check, the array has 5 million entries
(40 MB).

* 📦 **sending**: the array is copied into every task, `N` times. With 8
  workers that is 8 × 40 MB = 320 MB in total, twice the copying of
  Section 2, and more with more workers;
* 📚 **shared memory**: the array is copied into the block once, and every task
  receives only the small tuple.

🤔 *Guess before you run!*

In [ ]:
small = rng.standard_normal(5_000_000)  # 40 MB
shm_small = shared_memory.SharedMemory(create=True, size=small.nbytes)
shared_small = np.ndarray(small.shape, dtype=small.dtype, buffer=shm_small.buf)
shared_small[:] = small

# every task reads the whole array: start 0, stop at the end
whole = [(shm_small.name, small.shape, str(small.dtype), 0, small.size)] * N

with tasks.warm_pool(N) as pool:
    t_shared = %timeit -o -q -r 3 -n 1 list(pool.map(tasks.harmonics_shared, whole))
    t_sent = %timeit -o -q -r 3 -n 1 list(pool.map(tasks.harmonics_sent, [small] * N))

print(f"shared memory               {t_shared.best:5.2f} s")
print(f"sending the array {N} times  {t_sent.best:5.2f} s")

💡 **What happened?** Shared memory wins clearly: sending copies the 40 MB
array once per task, and all these copies cost more than the work itself.
With shared memory nothing is copied.

⚠️ **A limit.** Back to the whiteboard: pinning the document on the whiteboard
saves the photocopies, but every student still has to read the document. 📖 The same holds for the workers.
For heavy work, such as `harmonics`, reading the numbers is a small part of
the time. For light work, such as a plain sum, reading is almost all of it,
and the memory can deliver numbers to all the cores together only so fast, so
more workers do not help.

## 4. 🧹 Check the result and clean up

First we check that the parallel result agrees with the direct computation.
The order of the additions differs, so the comparison uses a tolerance
(`np.isclose`).

Then the blocks must be released, in two steps:

* `close()` detaches *this* process from the block, like closing a file;
* `unlink()` deletes the block itself, like deleting the file. Call it once,
  from the process that created the block.

⚠️ A block that is never unlinked keeps its memory until Python cleans up at
exit, or on some systems until the machine restarts.

In [ ]:
with tasks.warm_pool(N) as pool:
    total_shared = sum(pool.map(tasks.harmonics_shared, jobs_shared))

print("agrees:", np.isclose(total_shared, tasks.harmonics(data)))

for block in (shm, shm_small):
    block.close()
    block.unlink()
print("blocks released")

## 5. 🧵 One thread per worker

Some NumPy routines, such as the matrix product `a @ a`, call a linear algebra
library that starts **threads of its own**, usually one per core. Inside a
pool with one worker per core, on a laptop with 8 cores, that means
8 × 8 = 64 threads competing for 8 cores 🤯, and they slow each other down.
This is called *oversubscription*.

The usual fix is to give every worker **one** library thread. The libraries
read the number of threads from environment variables when NumPy is imported:

| Variable | Library |
|---|---|
| `OMP_NUM_THREADS` | anything built on OpenMP |
| `OPENBLAS_NUM_THREADS` | OpenBLAS, used by NumPy installed with pip on Windows, Linux, and Intel Macs |
| `MKL_NUM_THREADS` | Intel MKL, used by some conda installations |
| `VECLIB_MAXIMUM_THREADS` | Apple Accelerate, used by NumPy on Macs with Apple processors |

Setting all four does no harm: each library reads only its own.

NumPy is already imported in this notebook, so setting the variables now does
not change *this* process. It does change the **workers**: with spawn, a new
worker starts a fresh Python, which copies the environment variables of its
parent and then imports NumPy. The cell checks that a worker sees the
setting.

In [ ]:
thread_vars = (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
)

for var in thread_vars:
    os.environ[var] = "1"

with ProcessPoolExecutor(max_workers=2, mp_context=CTX) as pool:
    for var in thread_vars:
        print(
            f"a worker sees {var} =",
            pool.submit(os.getenv, var).result(),
        )

💡 The package `threadpoolctl` can also limit the threads from inside a
running process, with `with threadpool_limits(limits=1): ...`. It works for
OpenBLAS and MKL, so on most computers it finds one thread pool and prints its
library and its number of threads. Apple's Accelerate manages its threads
itself and is invisible to `threadpoolctl`: on a Mac with an Apple processor
the cell finds no pool, and only the environment variable helps. Run the
cell to see which case your computer is.

In [ ]:
from threadpoolctl import threadpool_info

for pool_info in threadpool_info():
    print(pool_info["internal_api"], "threads:", pool_info["num_threads"])
print("number of thread pools found:", len(threadpool_info()))

## 🎯 Summary

| Situation | Best choice |
|---|---|
| each worker needs a different piece, once | 📦 sending is fine |
| many tasks read the same large array | 📚 shared memory |
| workers call threaded libraries (`@`, `solve`, ...) | 🧵 one library thread per worker |

## 🧪 Try it yourself

1. In Section 3, make the job cheaper by calling `tasks.harmonics` with
   fewer terms (you will need a small helper in `tasks.py`). Does the gap
   between sending and shared memory grow or shrink? Why?
2. In Section 3, increase the array to 10 million entries. How do the two
   times change?
3. Comment out `shm.unlink()` and rerun the notebook. Python prints a warning
   when it exits. Read it: what does it tell you?